# 📡 Multi-Agent RAG : Bharti Airtel FY2025
### A Classroom-Ready Notebook: From Basic RAG to Hierarchical Multi-Agent Pipelines

---

## 🎯 What You Will Learn

| Section | Pattern | What Happens |
|---------|---------|-------------|
| 2 | Base RAG | One PDF → FAISS → Retriever → LLM |
| 3 | Network Multi-Agent | Research → Analysis → Writer (sequential chain) |
| 4 | Supervisor Multi-Agent | Supervisor routes queries to the right specialist |
| 5 | Hierarchical Multi-Agent | Parent coordinates Year-specific child agents |
| 6 | Hybrid (PDF + Web) | Merge live Screener.in data with historical PDF facts |

---

## 🧰 Tech Stack (2026-Current Packages)

```
langchain >= 0.3          langchain-community >= 0.3
langchain-groq >= 0.2     langchain-huggingface >= 0.1
langgraph >= 0.3          langgraph-supervisor >= 0.0.9
faiss-cpu >= 1.8          sentence-transformers >= 3.0
pypdf >= 4.0              beautifulsoup4 >= 4.12
requests >= 2.31          python-dotenv >= 1.0
```

## 📁 Knowledge Sources
1. **PDF** — `Bharti Airtel.pdf` (FY2024-25 Integrated Annual Report + AGM Notice)
2. **Live Web** — `https://www.screener.in/company/BHARTIARTL/consolidated/`

---
## Section 1 : Environment Setup

Install all required packages and load environment variables.  
We use **Groq** (free-tier, fast inference) with `llama-3.1-8b-instant` as the backbone LLM.

In [ ]:
# ─────────────────────────────────────────────────────────────
# 1.1  Install Dependencies (run once; restart kernel after)
# ─────────────────────────────────────────────────────────────
%pip install -q \
    langchain>=0.3 \
    langchain-community>=0.3 \
    langchain-groq>=0.2 \
    langchain-huggingface>=0.1 \
    langgraph>=0.3 \
    langgraph-supervisor>=0.0.9 \
    faiss-cpu>=1.8 \
    pypdf>=4.0 \
    sentence-transformers>=3.0 \
    beautifulsoup4>=4.12 \
    requests>=2.31 \
    python-dotenv>=1.0

print(" All packages installed. Restart the kernel if running for the first time.")

In [ ]:
# ─────────────────────────────────────────────────────────────
# 1.2  Environment Variables & Config
# ─────────────────────────────────────────────────────────────
import os
from dotenv import load_dotenv

# Load .env file (create one with GROQ_API_KEY=<your-key>)
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
if not GROQ_API_KEY:
    raise EnvironmentError(" GROQ_API_KEY not found. Add it to your .env file or set it manually.")

# ── PDF Path ─────────────────────────────────────────────────
PDF_PATH = "C:\\Users\\admin\\Desktop\\New_GenAI\\GenAI\\LangGraph\\Multi-Agent RAG\\Bharti Airtel.pdf"

# Verify PDF exists
if not os.path.exists(PDF_PATH):
    print(f"⚠️  PDF not found at: {PDF_PATH}")
    print("   Update PDF_PATH to your actual file location.")
else:
    print(f"✅ PDF found: {PDF_PATH}")

# ── Model Config ──────────────────────────────────────────────
# Groq model choices (2026 available models):
#   llama-3.1-8b-instant    → fastest, free tier
#   llama-3.3-70b-versatile → better reasoning
#   mixtral-8x7b-32768      → long context window
LLM_MODEL = "llama-3.3-70b-versatile"
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

print(f"\n LLM      : {LLM_MODEL} (via Groq)")
print(f" Embeddings: {EMBED_MODEL} (HuggingFace local)")

---
## Section 2 : Base RAG Pipeline

```
 PDF  ──► PyPDFLoader ──► TextSplitter ──► HuggingFace Embeddings
                                                    │
                                              FAISS VectorStore
                                                    │
                                              Retriever  ──► Groq LLM  ──► Answer
```

**Key concepts:**
- `PyPDFLoader` : reads pages from a PDF file
- `RecursiveCharacterTextSplitter` : breaks text into overlapping chunks
- `HuggingFaceEmbeddings` : turns text into vectors (runs locally, no API needed)
- `FAISS` : fast similarity search over vectors
- `ChatGroq` : LLM inference via Groq cloud

In [ ]:
pip install pypdf

In [ ]:
# ─────────────────────────────────────────────────────────────
# 2.1  Load PDF
# ─────────────────────────────────────────────────────────────
from langchain_community.document_loaders import PyPDFLoader

print("📄 Loading PDF...")
loader = PyPDFLoader(file_path=PDF_PATH)
raw_docs = loader.load()

print(f" Loaded {len(raw_docs)} pages from the annual report.")
print(f"\n📋 Sample (page 1 excerpt):")
print(raw_docs[0].page_content[:500])

In [ ]:
# ─────────────────────────────────────────────────────────────
# 2.2  Split Documents into Chunks
# ─────────────────────────────────────────────────────────────
from langchain_text_splitters import RecursiveCharacterTextSplitter


splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,        # each chunk ≈ 1000 characters
    chunk_overlap=200,      # 200-char overlap for context continuity
    length_function=len,
    separators=["\n\n", "\n", " ", ""]
)

chunks = splitter.split_documents(raw_docs)

print(f"✅ Split {len(raw_docs)} pages → {len(chunks)} chunks")
print(f"\n📦 Sample chunk:")
print(chunks[5].page_content[:300])

In [ ]:
# ─────────────────────────────────────────────────────────────
# 2.3  Create HuggingFace Embeddings + FAISS Vector Store
# ─────────────────────────────────────────────────────────────
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

print("⚙️  Loading HuggingFace embedding model (first run downloads ~90MB)...")
embeddings = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL,
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

print("🔢 Building FAISS index (vectorizing all chunks)...")
vectorstore = FAISS.from_documents(chunks, embeddings)

# Save locally for reuse (avoids re-embedding on every run)
vectorstore.save_local("airtel_faiss_index")

print(f"✅ FAISS index built with {vectorstore.index.ntotal} vectors.")
print("💾 Index saved to ./airtel_faiss_index/")

In [ ]:
# ─────────────────────────────────────────────────────────────
# 2.4  Build Retriever and LLM
# ─────────────────────────────────────────────────────────────
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Retriever: top-5 most relevant chunks
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

# Groq LLM
llm = ChatGroq(
    model=LLM_MODEL,
    temperature=0.3,
    api_key=GROQ_API_KEY
)

# RAG Prompt Template
rag_prompt = ChatPromptTemplate.from_template("""
You are a financial analyst assistant specializing in Bharti Airtel's annual reports.
Use ONLY the context below to answer the question. Be precise and cite relevant figures.

Context:
{context}

Question: {question}

Answer:
""")

def format_docs(docs):
    return "\n\n".join(
        f"[Page {doc.metadata.get('page', '?')}]\n{doc.page_content}"
        for doc in docs
    )

# Build the RAG chain (LangChain Expression Language)
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

print("RAG chain ready!")

In [ ]:
# ─────────────────────────────────────────────────────────────
# 2.5  Test the Base RAG Pipeline
# ─────────────────────────────────────────────────────────────
test_queries = [
    "Summarize the AGM notice from the annual report.",
    "What were Airtel's total revenues for FY2025?",
    "Who are the board members mentioned in the annual report?",
]

for q in test_queries:
    print(f"\n{'='*60}")
    print(f" => Query: {q}")
    print(f"{'='*60}")
    answer = rag_chain.invoke(q)
    print(answer)

---
## Section 3 : Multi-Agent Network RAG

Three agents collaborate in sequence:

```
User Query
    │
    ▼
┌──────────────────┐
│  Research Agent  │  ← Retrieves raw data from PDF
└────────┬─────────┘
         │ raw_findings
         ▼
┌──────────────────┐
│  Analysis Agent  │  ← Identifies trends, growth, ratios
└────────┬─────────┘
         │ analysis
         ▼
┌──────────────────┐
│   Writer Agent   │  ← Produces a student-friendly summary
└────────┬─────────┘
         │
         ▼
    Final Report
```

**Architecture**: LangGraph `StateGraph` with typed state and sequential edges.

In [ ]:
# ─────────────────────────────────────────────────────────────
# 3.1  Define Shared State Schema
# ─────────────────────────────────────────────────────────────
from typing import TypedDict, Optional
from langgraph.graph import StateGraph, START, END

class NetworkAgentState(TypedDict):
    query: str
    raw_findings: Optional[str]    # output from Research Agent
    analysis: Optional[str]        # output from Analysis Agent
    final_report: Optional[str]    # output from Writer Agent

print(" NetworkAgentState schema defined.")
print("   Fields: query → raw_findings → analysis → final_report")

In [ ]:
# ─────────────────────────────────────────────────────────────
# 3.2  Define Agent Node Functions
# ─────────────────────────────────────────────────────────────

# ── Research Agent ───────────────────────────────────────────
research_prompt = ChatPromptTemplate.from_template("""
You are a financial research agent. Retrieve and list factual data from the context.
Focus on numbers, dates, KPIs, and direct statements from the annual report.
Do NOT interpret — only extract facts.

Context:
{context}

Query: {query}

Raw Findings:
""")

def research_agent_node(state: NetworkAgentState) -> NetworkAgentState:
    """Retrieves relevant passages from the PDF and extracts raw facts."""
    print("  Research Agent: Retrieving from FAISS...")
    docs = retriever.invoke(state["query"])
    context = format_docs(docs)
    chain = research_prompt | llm | StrOutputParser()
    findings = chain.invoke({"context": context, "query": state["query"]})
    return {**state, "raw_findings": findings}

# ── Analysis Agent ────────────────────────────────────────────
analysis_prompt = ChatPromptTemplate.from_template("""
You are a financial analysis agent. Given the raw research findings below,
identify key trends, growth rates, ratios, and business insights.
Compare figures where possible and highlight what is significant for investors.

Raw Findings:
{raw_findings}

Original Query: {query}

Analysis:
""")

def analysis_agent_node(state: NetworkAgentState) -> NetworkAgentState:
    """Interprets raw findings to extract trends and insights."""
    print("  Analysis Agent: Interpreting trends...")
    chain = analysis_prompt | llm | StrOutputParser()
    analysis = chain.invoke({
        "raw_findings": state["raw_findings"],
        "query": state["query"]
    })
    return {**state, "analysis": analysis}

# ── Writer Agent ──────────────────────────────────────────────
writer_prompt = ChatPromptTemplate.from_template("""
You are a financial writer creating educational content for MBA students.
Using the analysis below, write a clear, structured summary that:
- Uses simple language (avoid jargon)
- Includes key numbers and what they mean
- Ends with 2-3 key takeaways
- Is suitable for a classroom presentation

Analysis:
{analysis}

Original Query: {query}

Student-Friendly Summary:
""")

def writer_agent_node(state: NetworkAgentState) -> NetworkAgentState:
    """Converts analysis into a student-friendly written report."""
    print("  Writer Agent: Composing final report...")
    chain = writer_prompt | llm | StrOutputParser()
    report = chain.invoke({
        "analysis": state["analysis"],
        "query": state["query"]
    })
    return {**state, "final_report": report}

print(" All three agent node functions defined.")

In [ ]:
# ─────────────────────────────────────────────────────────────
# 3.3  Build and Compile the Network Graph
# ─────────────────────────────────────────────────────────────
from langgraph.graph import StateGraph, START, END

network_builder = StateGraph(NetworkAgentState)

# Register nodes
network_builder.add_node("research_agent", research_agent_node)
network_builder.add_node("analysis_agent", analysis_agent_node)
network_builder.add_node("writer_agent",   writer_agent_node)

# Define sequential edges
network_builder.add_edge(START,             "research_agent")
network_builder.add_edge("research_agent",  "analysis_agent")
network_builder.add_edge("analysis_agent",  "writer_agent")
network_builder.add_edge("writer_agent",    END)

network_graph = network_builder.compile()

print("   Network Multi-Agent graph compiled.")
print("   Flow: START → research_agent → analysis_agent → writer_agent → END")

In [ ]:
network_graph

In [ ]:
# ─────────────────────────────────────────────────────────────
# 3.4  Run the Network RAG Pipeline
# ─────────────────────────────────────────────────────────────
query = "What is Bharti Airtel's revenue and EBITDA performance in FY2025?"

print(f"\n{'='*65}")
print(f" Query: {query}")
print(f"{'='*65}\n")
print(" Running 3-Agent Network Pipeline...\n")

result = network_graph.invoke({"query": query})

print("\n" + "─"*65)
print(" RAW FINDINGS (Research Agent)")
print("─"*65)
print(result["raw_findings"])

print("\n" + "─"*65)
print(" ANALYSIS (Analysis Agent)")
print("─"*65)
print(result["analysis"])

print("\n" + "─"*65)
print(" FINAL REPORT (Writer Agent)")
print("─"*65)
print(result["final_report"])

---
## Section 4 : Supervisor Multi-Agent RAG

The **Supervisor** pattern adds intelligent routing:  
instead of always going Research → Analysis → Writer, the supervisor decides which specialist to call.

```
                    ┌──────────────────────┐
User Query ──────►  │  Supervisor Agent    │
                    │  (LLM router)        │
                    └───┬──────┬──────┬───┘
                        │      │      │
                  Numbers  Trends  Summaries
                        │      │      │
                        ▼      ▼      ▼
                   Research Analysis Writer
                    Agent   Agent   Agent
```

We use `langgraph-supervisor` with `create_react_agent` and `create_supervisor` — the canonical 2026 API.

In [ ]:
# ─────────────────────────────────────────────────────────────
# 4.1  Define Tool Functions for Specialist Agents
# ─────────────────────────────────────────────────────────────
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent
from langgraph_supervisor import create_supervisor

# ── Tool: Retrieve financial data from PDF ────────────────────
@tool
def retrieve_financial_data(query: str) -> str:
    """
    Retrieve specific financial data, numbers, metrics, KPIs, and figures
    from Bharti Airtel's FY2025 Annual Report PDF.
    Use this for questions about revenue, EBITDA, capex, subscribers, etc.
    """
    docs = retriever.invoke(query)
    context = format_docs(docs)
    prompt = f"""Extract specific financial numbers and metrics for: {query}

Context from annual report:
{context}

List only the relevant numbers, figures, and KPIs:"""
    return llm.invoke(prompt).content

# ── Tool: Analyze trends ──────────────────────────────────────
@tool
def analyze_financial_trends(data: str) -> str:
    """
    Analyze financial trends, growth rates, year-over-year changes,
    and business performance patterns from Airtel data.
    Use this after gathering raw financial data to identify trends.
    """
    prompt = f"""Analyze the following financial data for trends, growth rates, 
and significant patterns. Provide percentage changes where calculable:

{data}

Trend Analysis:"""
    return llm.invoke(prompt).content

# ── Tool: Summarize for students ──────────────────────────────
@tool
def write_student_summary(content: str) -> str:
    """
    Write a clear, student-friendly summary of financial findings.
    Use this to produce final readable outputs for educational purposes.
    """
    prompt = f"""Write a clear, concise summary of the following for MBA students.
Use plain English. Include key numbers. End with 3 bullet-point takeaways:

{content}

Student Summary:"""
    return llm.invoke(prompt).content

print("   Three agent tools defined:")
print("   • retrieve_financial_data  → Research specialist")
print("   • analyze_financial_trends → Analysis specialist")
print("   • write_student_summary    → Writer specialist")

In [ ]:
# ─────────────────────────────────────────────────────────────
# 4.2  Create Specialist ReAct Agents
# ─────────────────────────────────────────────────────────────

# Research specialist — handles numbers and data retrieval
research_specialist = create_react_agent(
    model=llm,
    tools=[retrieve_financial_data],
    name="research_specialist",
    prompt=(
        "You are a financial research specialist. "
        "Your job is to retrieve accurate financial figures from Airtel's annual report. "
        "Always use the retrieve_financial_data tool. Be precise and cite page numbers."
    )
)

# Analysis specialist — handles trends and comparisons
analysis_specialist = create_react_agent(
    model=llm,
    tools=[retrieve_financial_data, analyze_financial_trends],
    name="analysis_specialist",
    prompt=(
        "You are a financial analysis specialist. "
        "First retrieve data, then analyze trends and growth patterns. "
        "Focus on year-over-year changes and business performance insights."
    )
)

# Writer specialist — handles summaries and reports
writer_specialist = create_react_agent(
    model=llm,
    tools=[retrieve_financial_data, write_student_summary],
    name="writer_specialist",
    prompt=(
        "You are an educational content writer. "
        "Retrieve relevant information and write clear summaries for students. "
        "Always produce structured, easy-to-read content with key takeaways."
    )
)

print(" Three specialist ReAct agents created.")

In [ ]:
# ─────────────────────────────────────────────────────────────
# 4.3  Create the Supervisor
# ─────────────────────────────────────────────────────────────

supervisor_prompt = """
You are a supervisor managing a team of Bharti Airtel financial analysts.
Route each query to the most appropriate specialist:

• research_specialist  → queries about specific numbers, figures, KPIs, revenues, profits
• analysis_specialist  → queries about trends, comparisons, growth, year-over-year changes
• writer_specialist    → queries asking for summaries, overviews, explanations, AGM notices

After the specialist responds, compile their output into a final answer.
If the query requires multiple agents, coordinate them sequentially.
"""

supervisor_workflow = create_supervisor(
    agents=[research_specialist, analysis_specialist, writer_specialist],
    model=llm,
    prompt=supervisor_prompt,
)

supervisor_app = supervisor_workflow.compile()

print("   Supervisor Multi-Agent system compiled.")
print("   Routing: research_specialist | analysis_specialist | writer_specialist")

In [ ]:
supervisor_app

In [ ]:
# ─────────────────────────────────────────────────────────────
# 4.4  Run Supervisor Multi-Agent Queries
# ─────────────────────────────────────────────────────────────
from langchain_core.messages import HumanMessage

supervisor_queries = [
    "Compare FY2025 revenue with FY2024 — show growth rates.",
    "Summarize the FY2025 AGM notice for students.",
    "What are Airtel's EBITDA margins and how have they trended?",
]

for query in supervisor_queries:
    print(f"\n{'='*65}")
    print(f" Query: {query}")
    print(f"{'='*65}")

    result = supervisor_app.invoke(
        {"messages": [HumanMessage(content=query)]},
        config={"recursion_limit": 20}
    )

    # Extract last assistant message as the final answer
    final_messages = [m for m in result["messages"] if hasattr(m, 'content')]
    print("\n Final Answer:")
    print(final_messages[-1].content)

In [ ]:
# ─────────────────────────────────────────────────────────────
# 4.3  Create the Supervisor (Strict Stop Condition)
# ─────────────────────────────────────────────────────────────
supervisor_prompt = """
You are a supervisor managing a team of Bharti Airtel financial analysts.
Route each query to the most appropriate specialist:

• research_specialist  → queries about specific numbers, figures, KPIs, revenues, profits
• analysis_specialist  → queries about trends, comparisons, growth, year-over-year changes
• writer_specialist    → queries asking for summaries, overviews, explanations, AGM notices

Important:
- Call only the necessary specialist(s).
- After producing the final answer, STOP and return it.
- Do not call agents again unless explicitly required.
- Never loop or re-route the same query repeatedly.
- Always end with a single assistant message containing the final answer.
"""

supervisor_workflow = create_supervisor(
    agents=[research_specialist, analysis_specialist, writer_specialist],
    model=llm,
    prompt=supervisor_prompt,
)

supervisor_app = supervisor_workflow.compile()

print(" Supervisor Multi-Agent system compiled with strict stop condition.")


# ─────────────────────────────────────────────────────────────
# 4.4  Run Supervisor Multi-Agent Queries (Safe Debug Version)
# ─────────────────────────────────────────────────────────────
from langchain_core.messages import HumanMessage

supervisor_queries = [
    "What are Airtel's EBITDA margins and how have they trended?",
]

for query in supervisor_queries:
    print(f"\n{'='*65}")
    print(f" Query: {query}")
    print(f"{'='*65}")

    try:
        result = supervisor_app.invoke(
            {"messages": [HumanMessage(content=query)]},
            config={"recursion_limit": 5}  # keep low for debugging
        )

        # Print all intermediate messages to see agent flow
        print("\n--- Debug: All Messages ---")
        for m in result["messages"]:
            if hasattr(m, 'content'):
                print(type(m).__name__, ":", m.content)

        # Extract last assistant message as the final answer
        final_messages = [m for m in result["messages"] if hasattr(m, 'content')]
        if final_messages:
            print("\n Final Answer:")
            print(final_messages[-1].content)
        else:
            print("\n No final answer produced. Supervisor may still be looping.")

    except Exception as e:
        print("\n Error during execution:", str(e))


---
## Section 5 : Hierarchical Multi-Agent RAG

A **parent supervisor** coordinates year-specific child agents:

```
User Query: "Summarize Airtel's 4-year performance"
        │
        ▼
┌─────────────────────────────────────┐
│        PARENT Supervisor Agent      │
└──┬──────┬──────┬──────┬─────────────┘
   │      │      │      │
   ▼      ▼      ▼      ▼
 FY22   FY23   FY24   FY25
Agent  Agent  Agent  Agent
   │      │      │      │
   └──────┴──────┴──────┘
              │
    Parent aggregates all results
              │
    Consolidated 4-Year Overview
```

**Note**: Since we have a single PDF containing all years, each child agent uses  
a year-specific retriever filter on metadata.

In [ ]:
# ─────────────────────────────────────────────────────────────
# 5.1  Create Year-Specific Tool Factory
# ─────────────────────────────────────────────────────────────

def make_year_tool(year: str):
    """
    Factory function that creates a retrieval tool focused on a specific fiscal year.
    Each child agent gets its own version of this tool.
    """
    @tool(name=f"retrieve_{year}_data")
    def year_tool(query: str) -> str:
        f"""
        Retrieve Bharti Airtel financial data specifically for {year}.
        Use for questions about {year} revenues, EBITDA, subscribers, and performance.
        """
        # Filter query to focus on the specific year
        year_query = f"{query} {year} fiscal year"
        docs = retriever.invoke(year_query)
        context = format_docs(docs)

        prompt = f"""From the annual report context, extract data specifically for {year}.
Focus on: revenue, EBITDA, net profit, subscribers, ARPU, capex.

Context:
{context}

Query: {query}

{year} Data Summary:"""
        return llm.invoke(prompt).content

    year_tool.__doc__ = f"Retrieve Bharti Airtel financial data specifically for {year}."
    return year_tool

# Create year-specific tools
tool_fy22 = make_year_tool("FY2022")
tool_fy23 = make_year_tool("FY2023")
tool_fy24 = make_year_tool("FY2024")
tool_fy25 = make_year_tool("FY2025")

print(" Year-specific tools created for FY2022, FY2023, FY2024, FY2025")

In [ ]:
# ─────────────────────────────────────────────────────────────
# 5.2  Create Child Agents (One per Fiscal Year)
# ─────────────────────────────────────────────────────────────

def make_year_agent(year: str, year_tool):
    """Create a ReAct agent specialised for a specific fiscal year."""
    return create_react_agent(
        model=llm,
        tools=[year_tool],
        name=f"agent_{year.lower()}",
        prompt=(
            f"You are a financial specialist for Bharti Airtel {year}. "
            f"Your ONLY job is to retrieve and report {year} financial data. "
            f"Always use the retrieve_{year}_data tool. "
            f"Be precise, cite numbers, and clearly label all figures as {year} data."
        )
    )

agent_fy22 = make_year_agent("FY2022", tool_fy22)
agent_fy23 = make_year_agent("FY2023", tool_fy23)
agent_fy24 = make_year_agent("FY2024", tool_fy24)
agent_fy25 = make_year_agent("FY2025", tool_fy25)

print("✅ Four year-specific child agents created.")

In [ ]:
# ─────────────────────────────────────────────────────────────
# 5.3  Create Parent Supervisor for Hierarchical RAG
# ─────────────────────────────────────────────────────────────

parent_supervisor_prompt = """
You are a senior financial analyst supervising a team of year-specific Airtel analysts.
Your team covers: agent_fy2022, agent_fy2023, agent_fy2024, agent_fy2025.

For multi-year queries:
1. Delegate to each relevant year's agent to gather their specific data
2. Wait for all agents to respond
3. Synthesize all responses into a coherent consolidated overview
4. Highlight year-over-year trends and the overall trajectory

Always produce a final consolidated answer after gathering all year data.
"""

hierarchical_workflow = create_supervisor(
    agents=[agent_fy22, agent_fy23, agent_fy24, agent_fy25],
    model=llm,
    prompt=parent_supervisor_prompt,
)

hierarchical_app = hierarchical_workflow.compile()

print("✅ Hierarchical Multi-Agent system compiled.")
print("   Parent → [FY2022 Agent | FY2023 Agent | FY2024 Agent | FY2025 Agent]")

In [ ]:
# ─────────────────────────────────────────────────────────────
# 5.4  Run Hierarchical Multi-Year Query
# ─────────────────────────────────────────────────────────────
hierarchical_query = (
    "Summarize Airtel's revenue and EBITDA performance across FY2022–FY2025. "
    "Show year-over-year growth and identify the most significant trends."
)

print(f"\n{'='*65}")
print(f"📨 Query: {hierarchical_query}")
print(f"{'='*65}")
print("\n🚀 Running Hierarchical Multi-Agent Pipeline...")
print("   (Each year agent will be called sequentially by the parent)\n")

hier_result = hierarchical_app.invoke(
    {"messages": [HumanMessage(content=hierarchical_query)]},
    config={"recursion_limit": 40}  # Higher limit for 4 sub-agents
)

print("\n" + "─"*65)
print("📋 CONSOLIDATED 4-YEAR OVERVIEW")
print("─"*65)
final_msgs = [m for m in hier_result["messages"] if hasattr(m, 'content')]
print(final_msgs[-1].content)

---
## Section 6 : Live Data Integration (Screener.in + PDF)

We combine:
- **Static**: Historical financials from the PDF (via RAG)
- **Dynamic**: Real-time stock price, market cap, P/E from Screener.in

```
                ┌─────────────────┐     ┌──────────────────┐
                │  PDF (RAG)      │     │  Screener.in     │
                │  - FY25 EPS     │     │  - Live Price    │
                │  - Revenue      │     │  - Market Cap    │
                │  - Net Profit   │     │  - P/E Ratio     │
                └────────┬────────┘     └────────┬─────────┘
                         │                       │
                         └──────────┬────────────┘
                                    ▼
                         Hybrid Analysis Agent
                                    │
                          Combined Insight Report
```

In [ ]:
# ─────────────────────────────────────────────────────────────
# 6.1  Scrape Live Data from Screener.in
# ─────────────────────────────────────────────────────────────
import requests
from bs4 import BeautifulSoup
import json
import time

SCREENER_URL = "https://www.screener.in/company/BHARTIARTL/consolidated/"

def scrape_screener_data(url: str) -> dict:
    """
    Scrape key financial metrics from Screener.in.
    Returns a dict with stock price, market cap, P/E and other ratios.
    """
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/120.0.0.0 Safari/537.36"
        ),
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
        "Accept-Language": "en-US,en;q=0.5",
        "Accept-Encoding": "gzip, deflate, br",
        "Connection": "keep-alive",
    }

    metrics = {}

    try:
        response = requests.get(url, headers=headers, timeout=15)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")

        # ── Company Name ────────────────────────────────────
        name_tag = soup.find("h1", {"class": "h2"}) or soup.find("h1")
        metrics["company_name"] = name_tag.text.strip() if name_tag else "Bharti Airtel"

        # ── Key Ratios (top summary section) ─────────────────
        ratio_section = soup.find("ul", {"id": "top-ratios"})
        if ratio_section:
            for li in ratio_section.find_all("li"):
                name_span = li.find("span", {"class": "name"})
                value_span = li.find("span", {"class": "number"})
                if name_span and value_span:
                    key = name_span.text.strip().replace(" ", "_").lower()
                    value = value_span.text.strip()
                    metrics[key] = value

        # ── Fallback: Extract from meta description ───────────
        if not metrics.get("market_cap"):
            meta = soup.find("meta", {"name": "description"})
            if meta:
                metrics["meta_description"] = meta.get("content", "")[:300]

        metrics["scrape_status"] = "success"
        metrics["source_url"] = url

    except requests.exceptions.RequestException as e:
        print(f" Network error: {e}")
        # Return simulated data for classroom use if scraping fails
        metrics = {
            "company_name": "Bharti Airtel Ltd",
            "current_price": "₹1,850 (simulated)",
            "market_cap": "₹11,05,000 Cr (simulated)",
            "stock_p/e": "72.5 (simulated)",
            "book_value": "₹148 (simulated)",
            "dividend_yield": "0.54% (simulated)",
            "roce": "13.2% (simulated)",
            "roe": "22.1% (simulated)",
            "scrape_status": "simulated_fallback",
            "source_url": url,
            "note": "Screener.in may block automated requests. Use VPN or manual lookup."
        }

    return metrics

# Fetch live data
print(" Fetching live data from Screener.in...")
live_data = scrape_screener_data(SCREENER_URL)

print(f"\n Live Market Data for Bharti Airtel:")
print(f"{'─'*45}")
for key, value in live_data.items():
    if key not in ["source_url", "scrape_status", "meta_description"]:
        print(f"  {key:<25} : {value}")
print(f"\n  Status : {live_data['scrape_status']}")
print(f"  Source : {live_data['source_url']}")

In [ ]:
# ─────────────────────────────────────────────────────────────
# 6.2  Hybrid Analysis — PDF History + Live Market Data
# ─────────────────────────────────────────────────────────────

def hybrid_query(
    question: str,
    live_metrics: dict,
    retriever,
    llm
) -> str:
    """
    Answer a question by combining:
    1. Historical PDF data (via RAG retriever)
    2. Live market data from Screener.in
    """
    # Step 1: Retrieve PDF context
    pdf_docs = retriever.invoke(question)
    pdf_context = format_docs(pdf_docs)

    # Step 2: Format live data as readable string
    live_context = "\n".join(
        f"  {k}: {v}" for k, v in live_metrics.items()
        if k not in ["source_url", "scrape_status", "meta_description"]
    )

    # Step 3: Build hybrid prompt
    hybrid_prompt_template = ChatPromptTemplate.from_template("""
You are a financial analyst combining historical annual report data with live market data.

═══ LIVE MARKET DATA (from Screener.in) ═══
{live_context}

═══ HISTORICAL DATA (from FY2025 Annual Report PDF) ═══
{pdf_context}

═══ QUESTION ═══
{question}

Provide a comprehensive answer that:
1. Quotes the current live metric(s)
2. Links them to relevant historical performance from the annual report
3. Gives an integrated investor perspective

Hybrid Answer:
""")

    chain = hybrid_prompt_template | llm | StrOutputParser()
    return chain.invoke({
        "live_context": live_context,
        "pdf_context": pdf_context,
        "question": question
    })

# ── Run Hybrid Queries ────────────────────────────────────────
hybrid_questions = [
    "What is Airtel's current stock price and how does it compare to FY2025 earnings?",
    "Given Airtel's current P/E ratio and FY2025 revenue growth, is the stock fairly valued?",
    "Compare Airtel's current market cap with its FY2025 reported revenues.",
]

for question in hybrid_questions:
    print(f"\n{'='*65}")
    print(f"📨 Hybrid Query: {question}")
    print(f"{'='*65}")
    answer = hybrid_query(question, live_data, retriever, llm)
    print(answer)

---
## Section 7 : Student Exercises

Now it's your turn! Use all the tools you've built in this notebook to answer the following questions.

### Instructions:
- For **each exercise**, choose the appropriate pipeline (base RAG, network, supervisor, hierarchical, or hybrid)
- Explain **why** you chose that pipeline
- Discuss what the output tells you as an **investor or analyst**

---
### Exercise 1 — AGM Notice Summary
*Best pipeline: Supervisor (writer_specialist) or Base RAG*

In [ ]:
# ─────────────────────────────────────────────────────────────
# Exercise 1: Summarize FY2025 AGM Notice
# ─────────────────────────────────────────────────────────────
# Try both approaches and compare the outputs!

agm_query = "Summarize the FY2025 Annual General Meeting (AGM) notice including key resolutions and dates."

# --- Approach A: Base RAG ----------------------------------------
print(" Approach A: Base RAG Pipeline")
print("─" * 50)
base_answer = rag_chain.invoke(agm_query)
print(base_answer)

# --- Approach B: Supervisor (Uncomment to use) ------------------
# print("\n Approach B: Supervisor Multi-Agent")
# print("─" * 50)
# sup_result = supervisor_app.invoke(
#     {"messages": [HumanMessage(content=agm_query)]},
#     config={"recursion_limit": 20}
# )
# msgs = [m for m in sup_result["messages"] if hasattr(m, 'content')]
# print(msgs[-1].content)

# ──────────────────────────────────────────────────────────────
#   YOUR REFLECTION (answer in a new cell below):
# Q1: Which approach gave a more complete summary? Why?
# Q2: What were the key resolutions voted on at the AGM?
# Q3: What does an AGM notice tell us about corporate governance?

### Exercise 2 : Revenue Growth FY2022–FY2025
*Best pipeline: Hierarchical Multi-Agent (requires multi-year data)*

In [ ]:
# ─────────────────────────────────────────────────────────────
# Exercise 2: 4-Year Revenue Growth Analysis
# ─────────────────────────────────────────────────────────────
revenue_query = (
    "Compare revenue growth across FY2022, FY2023, FY2024, and FY2025. "
    "Calculate CAGR if possible and identify the key drivers of growth in each year."
)

print(f" Query: {revenue_query}")
print("\n Running Hierarchical Multi-Agent Pipeline...\n")

rev_result = hierarchical_app.invoke(
    {"messages": [HumanMessage(content=revenue_query)]},
    config={"recursion_limit": 40}
)

final = [m for m in rev_result["messages"] if hasattr(m, 'content')]
print(final[-1].content)

# ──────────────────────────────────────────────────────────────
#  YOUR REFLECTION:
# Q1: What is Airtel's approximate revenue CAGR over 4 years?
# Q2: In which year was growth the highest? What drove it?
# Q3: How does Africa vs India segment performance differ?

### Exercise 3 : Current Market Position vs FY2025 Performance
*Best pipeline: Hybrid (combines live web data + PDF history)*

In [ ]:
# ─────────────────────────────────────────────────────────────
# Exercise 3: Market Position vs FY2025 Fundamentals
# ─────────────────────────────────────────────────────────────
market_query = (
    "Explain Airtel's current market position based on live stock data. "
    "How does the current valuation (P/E, market cap) reflect FY2025 business performance? "
    "Is the market pricing in future growth expectations?"
)

print(f" Query: {market_query}")
print("\n Running Hybrid Analysis (PDF + Live Data)...\n")

market_answer = hybrid_query(market_query, live_data, retriever, llm)
print(market_answer)

# ──────────────────────────────────────────────────────────────
#   YOUR REFLECTION:
# Q1: What does a high P/E ratio suggest about investor expectations?
# Q2: Does the current price seem justified given FY2025 earnings?
# Q3: What risks could affect this valuation going forward?

### Exercise 4 — Open-Ended Challenge 
*Choose any pipeline. Design your own question.*

In [ ]:
# ─────────────────────────────────────────────────────────────
# Exercise 4: Your Own Financial Analysis Question
# ─────────────────────────────────────────────────────────────
# Suggested topics to explore:
#   - Airtel's debt reduction strategy (FY2022-FY2025)
#   - 5G capital expenditure and subscriber growth
#   - Africa segment profitability vs India
#   - Dividend history and payout policy
#   - ARPU (Average Revenue Per User) trends

#   WRITE YOUR OWN QUERY HERE:
your_query = "YOUR QUESTION HERE — Replace this string!"

#   CHOOSE YOUR PIPELINE:
# Option 1: rag_chain.invoke(your_query)
# Option 2: network_graph.invoke({"query": your_query})
# Option 3: supervisor_app.invoke({"messages": [HumanMessage(content=your_query)]})
# Option 4: hierarchical_app.invoke({"messages": [HumanMessage(content=your_query)]})
# Option 5: hybrid_query(your_query, live_data, retriever, llm)

if your_query != "YOUR QUESTION HERE — Replace this string!":
    print(f" Your Query: {your_query}")
    answer = rag_chain.invoke(your_query)  # Replace with your chosen pipeline
    print("\n Answer:")
    print(answer)
else:
    print(" Replace `your_query` with your own financial question to get started!")
    print("\nTip: Try something like:")
    print("  'What is Airtel's ARPU trend from FY2022 to FY2025 and what drives it?'")

---
##  Summary & Architecture Comparison

| Pattern | When to Use | Pros | Cons |
|---------|------------|------|------|
| **Base RAG** | Single-hop factual Q&A | Fast, simple, cheap | No reasoning layer |
| **Network** | Always: Research → Analyze → Write | Structured, auditable | Rigid — no routing |
| **Supervisor** | Mixed query types | Flexible routing | Supervisor LLM cost |
| **Hierarchical** | Multi-source/multi-year | Scalable, parallel | Complex, higher latency |
| **Hybrid** | Live + historical analysis | Rich, real-world answers | Scraping fragility |

---

## Tech Stack Reference (2026 Versions)

```python
# Core
langchain >= 0.3                 # LCEL, prompts, chains
langchain-community >= 0.3       # PyPDFLoader, FAISS
langchain-groq >= 0.2            # ChatGroq LLM wrapper
langchain-huggingface >= 0.1     # HuggingFaceEmbeddings
langchain-core >= 0.3            # Tools, messages, runnables

# Agents
langgraph >= 0.3                 # StateGraph, START, END
langgraph-supervisor >= 0.0.9   # create_supervisor()

# ML / Vector
sentence-transformers >= 3.0    # all-MiniLM-L6-v2 embeddings
faiss-cpu >= 1.8                # Vector similarity search

# PDF / Web
pypdf >= 4.0                    # PDF text extraction
beautifulsoup4 >= 4.12          # HTML scraping
requests >= 2.31                # HTTP client
```

---
*Notebook by: Multi-Agent RAG Course | Built with LangGraph 2026 APIs*